### Step 1: Import Libraries & API Key

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr
import requests
import json
from IPython.display import display, Markdown

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

client = OpenAI()

### Step 2: Set up Pushover

Pushover is a tool that sends you a notification on your phone.

Step 2a -> Set up account in your browser
Step 2b -> Set up the app on your iPhone/Android; log into the same account
Step 2c -> In your browser, create an "Application/API Token"
Step 2d -> Copy your User Key (x) and API Token (y) into the .env file like this with your own keys:
PUSHOVER_USER = xxxxxxxxxxxxxxxxxxxx
PUSHOVER_TOKEN = yyyyyyyyyyyyyyyyyyy

**Then save changes to the .env file.**

Also run a test on https://pushover.net/ to manually send a notification from Pushover in your browser to pushover on your phone.

In [2]:
load_dotenv() #Rerun to load the changes you made (make sure you saved them (to the .env file) first)

True

In [3]:
# Get the values from the environment file (.env file):
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [7]:
# Check:
# Do this (in private!) to test that keys have been loaded.
# print(pushover_user)
# print(pushover_token)

In [4]:
# Create the send_notification function
def send_notification(message: str): #Created a function that expects a string
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data = payload)

In [6]:
#Test Run:
# send_notification("Hello to myself, from this amazing AI Engineering training!")

### Step 3: Describe Pushover as an LLM Tool

In [5]:
# LLM needs to know what tools it has and the details about each.
# We need to provide the LLM a description of the send_notification function using JSON.

# This is a dictionary that describes the send_notification function to the LLM
send_notification_function = {
    "name": "send_notification",  # tells LLM what this tool is and what it should be used for
    "description": "Sends a push notification to the user's phone via Pushover. Use this to \
        alert the user about important events, completed tasks, or time-sensitive information.", #description of what the function is

    "parameters": {
        "type": "object", #Required to write this because LLM's pass parameters through a JSON object
        # (inside which they'll be a string for this function (send_notification example))
        "properties": {
            "message": { #The send_notification function has the property of message.
                "type": "string", #The message is a string
                "description": "The notification message to send to the user's device" #What the message is
            }
        },
        "required": ["message"] #Required parameter is message.
        #This line puts the list of parameters that are required.
    }

}


### Step 4: Add Pushover to the List of Tools for the LLM

In [38]:
tools = [{"type": "function", "function": send_notification_function}] #We give the LLM the function description.
# List of function dictionaries for various tools, such as sending notifications.
# We have a dictionary (send_notification_function) inside of a list of dictionaries (tools)

### Step 5: Calling the Tool from an LLM

In [ ]:
# 1.Added a function to handle tools calls to enable Proper Tool Calling: (using the Tool Handling Code from previously)
def handle_tool_call(tool_calls): #passing a list of tool_calls
    tool_call = tool_calls[0] #Assuming just 1 tool call (thus the [0], which is the 1st element in the list of tool_calls)
    args = json.loads(tool_call.function.arguments) #takes the arguments from the tool_call function (so from send_notification)

    #To send the notification, i.e., call the tool
    send_notification(args['message']) #sends the message the user asks to send (verbatim) to my phone via Pushover
    #print(f"Sent notification: {args['message']}")
    #^Output would be: Sent notification: You're making amazing progress on the AI Engineering training by SuperDataScience. Keep up the great work!

    #Since there's only 1 tool call, we will have a dictionary (otherwise it would've been a list of tool_call_results):
    tool_call_result = {
        "role": "tool",
        "content": f"Notification sent: {args['message']}", #We decide what we want to tell the LLM in terms of what happened. This is tool dependent.
        #We take the message argument from the tool_call function
        #^We are telling the LLM the notification was sent. (Informing the LLM what happened)
        "tool_call_id": tool_call.id
    }
    #Return what to add to our "context" (i.e., messages list) (about tool call results), a dictionary

    return tool_call_result

In [8]:
# 2.Took messages out of the response variable so that the LLM can interact with it, rather than going straight from Gradio to the Code and then the Tool Call.
# So the messages list goes to the LLM.
messages = [
    {"role":"user", "content": "Please send me a notification telling me what amazing progress\
     I'm making on the AI Engineering training by SuperDataScience."} #User Message. User gives LLM instructions to call a tool.
    # Telling the LLM explicitly to send us a notification.
    ]

response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = messages,
    tools = tools #Pass the tools the LLM can use, so the tools list
    #tool_choice = "auto" (Default) #Lets the LLM choose the tool on its own. If tool_choice = "none", then the LLM won't call any tools.
    # If tool_choice = "required", then it would have to call at least 1 tool. Or alternatively, you could give it a dictionary of tools it could choose from.
)

message = response.choices[0].message

# Check if model wants to call a tool:
if message.tool_calls: #If there is a tool call: 
    # We process the tool (using tool_result),
    # Tell the LLM about what happened (using messages.append(message) and messages.append(tool_result)), and then
    # LLM tells us its new response.

    # 3.The LLM handles the tool call by calling the tool using the code and function handle_tool_call
    tool_result = handle_tool_call(message.tool_calls) #whole list of tool calls on purpose
    # (rather than using message.tool_calls[0] or [1] or [2] and have the code loop through them in the handle_tool_call
    # function instead - cleaner that way)
    # Pass the tool_calls into the function handle_tool_call to get the tool_result

    messages.append(message) #Want to give the LLM the message, so add message to "context", i.e. messages
    # 4.Message gets added to the messages list
    messages.append(tool_result) #Want to give the LLM the tool_result, so add info about the tool call response to
    # our "context", i.e. messages
    # 5.Info about the tool call response from our code gets added to our messages

    # 6.Now we pass all this context to the LLM:
    response = client.chat.completions.create( #.... invoke the LLM one more time to get its updated response
        #So now the LLM is involved in generating the response
        # 6.The LLM generates a response based on the tool call response that the code came up with.
        model = "gpt-4.1-mini",
        messages = messages
        #tools = tools #<- We will add this in the future (done when there is multiple tool calls)
    )    
    #print(message.content) #From the new LLM response.... print(message.content)

#else: #If there is no tool call, the LLM just prints the initial message = response.choices[0].message.content

    #print(message.content) #If the model isn't calling a tool, we just print what we got as a response.

    message = response.choices[0].message

print(message.content) #Print this regardless (will be either the new LLM response after a tool call or the LLM response without a tool call
# (whichever one is applicable))

I've sent you a notification to celebrate the amazing progress you're making on the AI Engineering training by SuperDataScience. Keep up the fantastic work!


*We get the following output from the LLM:*
*I've sent you a notification to celebrate the amazing progress you're making on the AI* *Engineering training by SuperDataScience. Keep up the fantastic work!*

***Output: I've sent you a notification celebrating the amazing progress you're making on the AI Engineering training by SuperDataScience. Keep up the great work!**

**So we get the notification on our phone through Pushover and the LLM responds in natural language (instead of saying what the code said: "Notification sent: ...")**

**Recall:** You have to process in your **code** what to do if the LLM does do a tool call because it's up to your code to catch that.

In [ ]:
# Check if model wants to call a tool
#print(message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_eewMZu8nlpAE380jNOtYLwLC', function=Function(arguments='{"message":"You are making amazing progress on the AI Engineering training by SuperDataScience. Keep up the great work!"}', name='send_notification'), type='function')])


*Output:*
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_eewMZu8nlpAE380jNOtYLwLC', function=Function(arguments='{"message":"You are making amazing progress on the AI Engineering training by SuperDataScience. Keep up the great work!"}', name='send_notification'), type='function')])

So the LLM is trying to call a tool as we have **tool_calls**=[ChatCompletionMessageFunctionToolCall(**id**='call_eewMZu8nlpAE380jNOtYLwLC', **function**=Function(**arguments**='{**"message":"You are making amazing progress on the AI Engineering training by SuperDataScience. Keep up the great work!"**}', **name='send_notification'**), type='function')]

We get an AI-generated notification message on our phone!

In [9]:
# 2.Took messages out of the response variable so that the LLM can interact with it, rather than going straight from Gradio to the Code and then the Tool Call.
# So the messages list goes to the LLM.
messages = [
    {"role":"user", "content": "Please send me 2 separate notifications: 1) Tell me what amazing progress\
     I'm making on the AI Engineering training by SuperDataScience.\
     and 2) Hey there!"} #User Message. User gives LLM instructions to call a tool.
    # Telling the LLM explicitly to send us a notification.
    ]

response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = messages,
    tools = tools #Pass the tools the LLM can use, so the tools list
    #tool_choice = "auto" (Default) #Lets the LLM choose the tool on its own. If tool_choice = "none", then the LLM won't call any tools.
    # If tool_choice = "required", then it would have to call at least 1 tool. Or alternatively, you could give it a dictionary of tools it could choose from.
)

message = response.choices[0].message

# Check if model wants to call a tool:
if message.tool_calls: #If there is a tool call: 
    # We process the tool (using tool_result),
    # Tell the LLM about what happened (using messages.append(message) and messages.append(tool_result)), and then
    # LLM tells us its new response.

    # 3.The LLM handles the tool call by calling the tool using the code and function handle_tool_call
    tool_result = handle_tool_call(message.tool_calls) #whole list of tool calls on purpose
    # (rather than using message.tool_calls[0] or [1] or [2] and have the code loop through them in the handle_tool_call
    # function instead - cleaner that way)
    # Pass the tool_calls into the function handle_tool_call to get the tool_result

    messages.append(message) #Want to give the LLM the message, so add message to "context", i.e. messages
    # 4.Message gets added to the messages list
    messages.append(tool_result) #Want to give the LLM the tool_result, so add info about the tool call response to
    # our "context", i.e. messages
    # 5.Info about the tool call response from our code gets added to our messages

    # 6.Now we pass all this context to the LLM:
    response = client.chat.completions.create( #.... invoke the LLM one more time to get its updated response
        #So now the LLM is involved in generating the response
        # 6.The LLM generates a response based on the tool call response that the code came up with.
        model = "gpt-4.1-mini",
        messages = messages
        #tools = tools #<- We will add this in the future (done when there is multiple tool calls)
    )    
    #print(message.content) #From the new LLM response.... print(message.content)

#else: #If there is no tool call, the LLM just prints the initial message = response.choices[0].message.content

    #print(message.content) #If the model isn't calling a tool, we just print what we got as a response.

    message = response.choices[0].message

print(message.content) #Print this regardless (will be either the new LLM response after a tool call or the LLM response without a tool call
# (whichever one is applicable))

BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_Q3GrcqR8fpWEP88VUaLBm5Ej", 'type': 'invalid_request_error', 'param': 'messages', 'code': None}}

We get the 1st message as a notification via Pushover but not the 2nd message at all. We also get the following Error: 

BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_Q3GrcqR8fpWEP88VUaLBm5Ej", 'type': 'invalid_request_error', 'param': 'messages', 'code': None}}

Which is basically saying that:

We called the LLM using the first Chat Completions line of code and got some response.
It may wanted to call tools. We handled the tool call. Then we added to the context window (the message (the tool call needed) and the tool call results). Then we called the assistant again with the 2nd Chat Completions code.

The LLM sees that we asked it to call 2 tools, but we only gave it results for one (through the handle_tool_call function (tool_call = tool_calls[0])).


#### Multiple Tool Calling Adjustment:

In [31]:
#Defined a function to handle Multiple tool calling with the same function (send_notification and with other functions/tools in brackets):

def handle_tool_call(tool_calls):  # passing a list of tool_calls
    tool_results = []
    for tool_call in tool_calls:  # loop over the actual tool_calls list
        function_name = tool_call.function.name #name of the function that we can call
        args = json.loads(tool_call.function.arguments)  # parse args per tool call
        #print(f"Calling function: {function_name}") #For future debugging :)

        #Route to the appropriate function based on function_name
        if function_name == "send_notification": 
            send_notification(args['message']) #send the notification, i.e., call the tool
            content = f"Notification sent: {args['message']}"
        #elif function_name == "insert_function_name_2":
        #   content = insert_function_name_2(arg['message']) #Assuming the function has a return statement

        #elif function_name == "insert_function_name_3":
        #   content = insert_function_name_3(arg['message']) #Assuming the function has a return statement

        #...
        else:
            content = f"Unknown Function: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id  # use this specific tool_call's id
        }
        tool_results.append(tool_call_result) #Adding tool_call_result dictionary to the tool_results list of dictionaries

    return tool_results  # return list of dicts (you'll need to append each one to messages)

In [21]:
messages = [
    {"role":"user", "content": "Please send me 2 separate notifications: 1) Tell me what amazing progress\
     I'm making on the AI Engineering training by SuperDataScience.\
     and 2) Hey there!"} #User Message. User gives LLM instructions to call a tool.
    # Telling the LLM explicitly to send us a notification.
    ]

# We will call the Pushover function twice to send 2 notifications, so tool_results will have 2 dictionaries with the 2 tool calls.

response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = messages,
    tools = tools #Pass the tools the LLM can use, so the tools list
)

message = response.choices[0].message
print(message)

# Check if model wants to call a tool:
if message.tool_calls: #If there is a tool call
    tool_result = handle_tool_call(message.tool_calls)
    messages.append(message) #append adds 1 element to a list
    print(messages)
    messages.extend(tool_result) #Changed from append() to extend() when we switched to multiple tool call handling

    #extend adds a list to another list; extend adds each tool_result dictionary as its own message
    #Search extend vs append python on Google images to see the difference.
    #In our case, we are adding list of dictionaries (tool_result) to the messages list.
    # So we need extend, since we are adding a list to another list. Thus, we must use extend and not append.
    # Extend merges these 2 lists by adding tool_result to the last entry in the messages list

    #if we used append, then we would get a list inside of list, instead of just merging them together like extend does.

    print(messages)
                    
# Now give this info to the LLM to respond to the user:
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages,
        tools = tools
    )

    message = response.choices[0].message

print(message.content) #LLM's response to the user after sending the push notifications to me.


ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_6uqEZiMSPuTarafprmsT4Tgu', function=Function(arguments='{"message": "Amazing progress alert! Keep up the fantastic work on the AI Engineering training by SuperDataScience. You\'re doing great!"}', name='send_notification'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_tRfykL6OCo7nzlMmbJ7PAiUo', function=Function(arguments='{"message": "Hey there!"}', name='send_notification'), type='function')])
[{'role': 'user', 'content': "Please send me 2 separate notifications: 1) Tell me what amazing progress     I'm making on the AI Engineering training by SuperDataScience.     and 2) Hey there!"}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_6uqEZiMSPuTarafprmsT4Tgu', function

Got the 2 notifications separately, one after the other in order, and the following message from the LLM:
I have sent you the two notifications: one about your amazing progress on the AI Engineering training by SuperDataScience, and the other simply saying "Hey there!"

### Step 2b, 3b, and 4b: Create a New Function, Describe it, and Add it to the List of Tools

In [39]:
# Dice Roll Function
import random

# Step 2b: Define the function
#Simulates rolling a 6-sided die
def dice_roll():
    result = random.randint(1,6)
    return result

# Step 3b: Describe the function in JSON for the LLM
roll_dice_function = {
    "name": "dice_roll",
    "description": "Simulates rolling a sinlge 6-sided die and returns the result. Use this when the user wants to roll a die for games, decisions, or random number generation.",
    "parameters": {
        "type": "object", #we have to keep this line for JSON description
        "properties": {}, #have to keep this line empty so the LLM knows there are no properties
        "required": [] #there's nothing required because there are no properties
    }
}

# Step 4b: Add function to list of tools for the LLM to call
tools.append({"type": "function", "function": roll_dice_function})

#append because are the dictionary as it appears above into the tools list (want to keep it as a dictionary and just add it to the end of the tools dictionary)
#if we were combining 2 lists together (ie, merging 2 lists), then we would use extend

In [40]:
# Tool Handling:

def handle_tool_call(tool_calls):  # passing a list of tool_calls
    # Loop over the tool_calls:
    tool_results = []
    
    for tool_call in tool_calls:  # loop over the actual tool_calls list
        function_name = tool_call.function.name #name of the function that we can call
        args = json.loads(tool_call.function.arguments)  # parse args per tool call
        #print(f"Calling function: {function_name}") #For future debugging :)

        #Route to the appropriate function based on function_name
        if function_name == "send_notification": 
            send_notification(args['message']) #send the notification, i.e., call the tool
            content = f"Notification sent: {args['message']}"
            
        elif function_name == "dice_roll": #since we need to return the function as text. Rolled: 6 (for example)
            content = f"Rolled: {dice_roll()}" #Assuming the function has a return statement

        #elif function_name == "insert_function_name_3":
        #   content = insert_function_name_3(arg['message']) #Assuming the function has a return statement

        #...
        else:
            content = f"Unknown Function: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id  # use this specific tool_call's id
        }
        tool_results.append(tool_call_result) #Adding tool_call_result dictionary to the tool_results list of dictionaries

    return tool_results  # return list of dicts (you'll need to append each one to messages)

In [ ]:
# Carry Out the Tool Calls
messages = [
    {"role":"user", "content": "Please do 2 things:\
     1) I'd like to roll 4 dice and\
     2) Send me a notification saying Hey There!"} # 2) Send me a notification with the highest of the rolls. #No notification gets sent.
    ]


response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = messages,
    tools = tools #Pass the tools the LLM can use, so the tools list
)

message = response.choices[0].message

# Check if model wants to call a tool:
if message.tool_calls: #If there is a tool call
    from pprint import pprint
    pprint(message.tool_calls) # Output shows LLM is calling 5 tools: 4 dice_rolls and a send_notification
    tool_result = handle_tool_call(message.tool_calls)
    messages.append(message) #append adds 1 element to a list
    messages.extend(tool_result) #Changed from append() to extend() when we switched to multiple tool call handling

# Now give this info to the LLM to respond to the user:
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages,
        tools = tools
    )

    message = response.choices[0].message

print(message.content) #LLM's response to the user after sending the push notifications to me.

[ChatCompletionMessageFunctionToolCall(id='call_4z6ZBRPjmpoczCiOVfdETUXQ', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_lM0e7BiNKOeW1db30Dep5rSq', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_6LdBDgZ2U1epzo8DJEGV2fVq', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_aEdl7datkR3UvyNF2TrFQZua', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_r0OlV68Vd7JHWDXk2zKu8Xbj', function=Function(arguments='{"message": "Hey There!"}', name='send_notification'), type='function')]
Here are the results of your 4 dice rolls: 6, 3, 3, and 4. Also, I've sent you a notification saying "Hey There!" to your device. Let me know if you need anything else!


*Output:*

[ChatCompletionMessageFunctionToolCall(id='call_mO3vCCzIjc0hqH3wxMpe8vBj', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_VWp12x54YQY5GkFhCJL4etBq', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_rnTP1zdQfEpRCvyphktgCwKl', function=Function(arguments='{"message": "Hey There!"}', name='send_notification'), type='function')]
I rolled two dice for you with results 2 and 4. Also, I have sent you a notification saying "Hey There!" Is there anything else you'd like to do?

In [ ]:
# This one doesn't send the notification...
messages = [
    {"role":"user", "content": "Please do 2 things:\
     1) I'd like to roll 2 dice and\
     2) Send me a notification with the highest of the rolls."} # 2) Send me a notification with the highest of the rolls. #No notification gets sent.
    ]


response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = messages,
    tools = tools #Pass the tools the LLM can use, so the tools list
)

message = response.choices[0].message

# Check if model wants to call a tool:
if message.tool_calls: #If there is a tool call
    from pprint import pprint
    pprint(message.tool_calls) # Output shows LLM is calling 3 tools: 2 dice rolls and a send notification
    tool_result = handle_tool_call(message.tool_calls)
    messages.append(message) #append adds 1 element to a list
    messages.extend(tool_result) #Changed from append() to extend() when we switched to multiple tool call handling

# Now give this info to the LLM to respond to the user:
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages,
        tools = tools
    )

    message = response.choices[0].message

print(message.content) #LLM's response to the user after sending the push notifications to me.

[ChatCompletionMessageFunctionToolCall(id='call_erMQBDxI0DuXM84jyNGX0dTL', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_7h9NwA0v2YyFFaCoHzo494S6', function=Function(arguments='{}', name='dice_roll'), type='function')]
None


In [ ]:
# Fixed the above using while loop instead of a for loop:

messages = [
    {"role":"user", "content": "Please do 2 things:\
     1) I'd like to roll 2 dice and\
     2) Send me a notification with the highest of the rolls."} # 2) Send me a notification with the highest of the rolls. #No notification gets sent.
    ]


response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = messages,
    tools = tools #Pass the tools the LLM can use, so the tools list
)

message = response.choices[0].message

# Check if model wants to call a tool:
while message.tool_calls: #Need a while loop so that if there is another consecutive tool call, we can use handle_tool_call again (and get the push notification)
    from pprint import pprint
    pprint(message.tool_calls) # Output shows LLM is calling 3 tools: 2 dice rolls and a send notification
    tool_result = handle_tool_call(message.tool_calls)
    messages.append(message) #append adds 1 element to a list
    messages.extend(tool_result) #Changed from append() to extend() when we switched to multiple tool call handling

# Now give this info to the LLM to respond to the user:
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages,
        tools = tools #for consecutive tool calls
    )

    message = response.choices[0].message

    #Note: Consider adding protection from infinite consecutive tool calling (from the while loop)

print(message.content) #LLM's response to the user after sending the push notifications to me.

[ChatCompletionMessageFunctionToolCall(id='call_ZZRXIKkmsAs5mpUQmdVFvhhh', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_TVKzXjlDl3ddp8Ne8izyn5cQ', function=Function(arguments='{}', name='dice_roll'), type='function')]
[ChatCompletionMessageFunctionToolCall(id='call_wGyRAq0NUHWFJbRS73cVq64L', function=Function(arguments='{"message":"The highest roll of the two dice is 3."}', name='send_notification'), type='function')]
I rolled two dice and got 3 on each. I've sent you a notification with the highest roll, which is 3. If you want to try again or do anything else, just let me know!


#### So this template can handle tool calls properly, process multiple tools at the same time, and do consecutive tool calls as well.